# PIR UE9 - Séance 6 :

## __Transcriptomique : Analyses exploratoires et analyse d'expression différentielle__

Sandrine Caburet

*d'après plusieurs TPs d'analyses bioinformatique du M1 Génétique,  
avec des données générées par Yves Clément* 

**Tutoriel**

Nous allons ici analyser des données de transcriptomique de type "bulk RNAseq", obtenus à partir d'échantillons de cortex et de foie de souris adultes.

Les souris de ce dataset sont issues de 3 lignées differentes : Wild Type (WT), Duplication (Dup) et Deletion (Del) d'une région du chromosome 7 murin synténique d'une région humaine impliquée dans l'autisme. Les échantillons de cortex ont fait l'objet d'une première étude en 2014 : Blumenthal et al. (AJHG 2014, DOI : 10.1016/j.ajhg.2014.05.004).

Ces données d'échantillons de cortex font partie d'un plus gros dataset souris qui a été publié en 2018 sur GEO (GSE76872) et dans SRA (PRJNA312352), qui a ensuite été utilisé pour les analyses présentées dans l'article Tai et al. (AJHG, 2022, 10.1016/j.ajhg.2022.08.012).

Les données disponibles regroupent tous les tissus étudiés (Cortex, Striatum (Corps strié), Cerebellum (Cervelet), Foie, Graisse blanche, Graisse brune) dans 16 souris : 8 WT, 4 Dup et 4 Del.

Nous allons analyser les données des RNAseq de foie et de cortex des souris Wild_Type.

Pour démarrer le pipeline d'analyse de ces données, notre collègue Yves Clément a préalablement récupéré les données de séquences brutes (raw reads) des échantillons. Tous les échantillons on été ré-analysés dans le même pipeline NF-Core récent, suivant ces différentes étapes : 
 - vérification des reads avec FastQC
 - préparation des reads avec Fastp
 - mapping sur le génome de référence de souris GRCm39 avec Star
 - comptage des reads à partir des bam avec Salmon (gene level) 
 
Nous allons donc partir de ces comptages de reads pour poursuivre l'analyse.  

**Dans ce tutoriel, nous allons voir comment :**  
 **- sélectionner des données correspondant à certains échantillons dans un gros dataset**   
 **- réaliser une analyse exploratoire des données de transcriptomique de type "bulk RNAseq"**   
 **- réaliser une analyse d'expression différentielle**   
 **- visualiser les résultats pour identifier les gènes DE les plus intéressants**  

## 0. Set up parameters and RSession
---

We start with setting up paths to directories.

In [ ]:
## Code cell 1 ##

#setwd("~/meg_m1_gb_r")
myworking_path <- "./" # modify as necessary


We install required libraries and load them.

In [ ]:
## Code cell 2 ##

# list the required libraries
requiredLib <- c(
    "statmod",
    "RColorBrewer",
    "dendextend",
    "stats",
    "grDevices",
    "BiocManager",
    "writexl",
    "ggnewscale",
    "ggupset",
    "writexl",
    "readxl",
    "ggridges",
    "MatrixGenerics",
    "gplots",
    "ggplot2",
    "ggrepel",
    "ggfortify")
requiredBiocLib <- c("DESeq2", 
                     "org.Mm.eg.db",
                     "clusterProfiler",
                     "enrichplot",
                     "ComplexHeatmap",
                     "ReactomePA",
                     "GenomicRanges",
                     "GenomeInfoDb",
                     "SummarizedExperiment",
                     "matrixStats",
                     "fgsea",
                     "data.table",
                     "sva")


# install required libraries if not
for (lib in requiredLib) {
  if (!require(lib, character.only = TRUE, quiet = TRUE)) {
    install.packages(lib, quiet = TRUE)
  }
}

for( lib in requiredBiocLib) {
  if (!require(lib, character.only = TRUE, quiet = TRUE)) {
  BiocManager::install(lib, quiet = TRUE)
  }
}

# load libraries
message("Loading required libraries")
for (lib in requiredLib) {
  library(lib, character.only = TRUE)}

for (lib in requiredBiocLib) {
  library(lib, character.only = TRUE)}


rm(lib, requiredLib, requiredBiocLib)

<div class="alert alert-block alert-warning"> <b> En cas d'erreur pour l'installations des librairies R : </b><br>
Si vous obtenez une erreur du type "path not writable", c'est parce que la commande n'arrive pas à trouver le dossier nécessaire dans votre home.  <br>   
Pour résoudre ce problème, passez la cellule 2bis suivante en code, lancez là, puis relancez la cellule de code 2.     
</div>

Et enfin, bonne pratique à toujours faire, nous gardons une trace des librairies chargées dans notre session :

In [ ]:
## Code cell 3 ##

# keep tracks of R and packages for this study

sessionInfo()

## 1. Chargement des Data
---


**Data Cortex et Foie - Souris - Blumenthal-2014/Tai-2022**

Article Tai et al. (AJHG, 2022, 10.1016/j.ajhg.2022.08.012).    

Les données disponibles regroupent tous les tissus étudiés (Cortex, Striatum (Corps strié), Cerebellum (Cervelet), Foie, Graisse blanche, Graisse brune) dans 16 souris : 8 WT, 4 Dup et 4 Del. 

Nous allons récupérer les données des RNAseq de foie et de cortex, uniquement pour les souris Wild_Type.

### 1.A - Metadata Blumenthal-2014/Tai-2022

Les metadata sont obtenues depuis SRA (PRJNA312352, info accessible en partant par exemple de la page GEO dataset correspondant au projet : GSE76872), et en sélectionnant les échantillons qui nous intéressent.   
Ici un fichier global a été récupéré depuis SRA Run Selector. Il contient les informations sur les 96 échantillons (16 souris x 6 tissus). 

In [ ]:
## Code cell 4 ##

# Chargement des métadonnées
metadataTai <- read.table("/srv/data/meg-m1-gb/RNAseq/Tai-2022/SraRunTable-mmu-Blumenthal2014.csv",
                       header = TRUE, sep = ",", stringsAsFactors = FALSE)

# Afficher les premières lignes
str(metadataTai)
dim(metadataTai)
head(metadataTai, 4)

On voit que les infos qui nous intéressent pour le tri sont dans les colonnes : 
 - genotype pour la condition
 - tissue ou source_name pour le tissu d'intérêt
 - animal_id pour le numéro de la souris
 - gender pour le sexe de l'animal

On commence par ne récuperer que les lignes qui nous intéressent : 

In [ ]:
## Code cell 5 ##

# Filtrer les lignes où 'genotype' est "Wild Type" et 'source_name' est "Cortex" ou "Liver"
metadataTai_wt <- metadataTai[
  metadataTai$genotype == "Wild Type" & (metadataTai$source_name %in% c("Cortex", "Liver")),
  ]
head(metadataTai_wt, 4)


On ajoute une colonne avec un nom plus facile à gérer, qui prend en compte le tissu, le sexe et l'identifiant de la souris.

In [ ]:
## Code cell 6 ##

# Créer la colonne sample_name basée sur le design expérimental
metadataTai_wt$sample_name <- with(metadataTai_wt, {
  # Déterminer le tissu 
  tissue <- ifelse(source_name == "Cortex", "Cortex",
            ifelse(source_name == "Liver", "Liver", NA))
  
  # Déterminer l'initiale du sexe
  g <- ifelse(tolower(gender) == "male", "m",
       ifelse(tolower(gender) == "female", "f", NA))
  
  # Déterminer le numéro de réplicat (basé sur erccmix)
  rep <- as.integer(ave(Experiment, tissue, g,
                        FUN = function(x) rank(x, ties.method = "first")))
  
  # Créer le nom de l'échantillon
  paste0(tissue, "_", g, rep)
})

# Placer sample_name en première colonne
metadataTai_wt <- metadataTai_wt[, c("sample_name", setdiff(names(metadataTai_wt), "sample_name"))]

# Renommer "Wild Type" en "Wild_Type"
metadataTai_wt$genotype <- factor(gsub("Wild Type", "Wild_Type", metadataTai_wt$genotype))                    
                    
head(metadataTai_wt, 3)

On vérifie rapidement que chaque animal (par exemple f1 ou m2) est bien présent 2 fois, une fois pour le Cortex et une seconde fois pour le foie : 

In [ ]:
## Code cell 7 ##

metadataTai_wt$sample_name

### 1.B - Counts data de Blumenthal-2014/Tai-2022

Les données de reads counts sont dans un fichier produit par Salmon à partir des bam générés par Star : "salmon.merged.gene_counts.tsv".   


In [ ]:
## Code cell 8 ##

# Chargement des données
countsTai <- read.table("/srv/data/meg-m1-gb/RNAseq/Tai-2022/mappingResults/star_salmon/salmon.merged.gene_counts.tsv",
                       header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# Afficher les premières lignes
str(countsTai)
head(countsTai,3)
dim(countsTai)

On extrait les colonnes qui nous intéressent pour n'avoir que les counts des échantillons WT, c'est à dire ceux dont les noms sont dans `metadataTai_wt`.

In [ ]:
## Code cell 9 ##

countsTai_wt <- subset(countsTai, select = metadataTai_wt$Experiment)

# rétablissement des valeurs de read counts sous la forme d'entiers
countsTai_wt[] <- lapply(countsTai_wt, as.integer)

On renomme les échantillons pour avoir des noms plus faciles à gérer : 

In [ ]:
## Code cell 10 ##

# Créer un vecteur de correspondance entre Experiment et sample_name depuis metadataTai_wt
names <- setNames(metadataTai_wt$sample_name, metadataTai_wt$Experiment)
names

# Renommer les colonnes de countsTai_wt en utilisant ce vecteur
colnames(countsTai_wt) <- names[colnames(countsTai_wt)]



On remet les noms de gènes devant les reads counts : 

In [ ]:
## Code cell 11 ##

countsTai_wt <- cbind(countsTai[, 1:2], countsTai_wt)

In [ ]:
## Code cell 12 ##

dim(countsTai_wt)
head(countsTai_wt, 3)

### 1.C - Fichier de metadata pour notre analyse

On crée un dataframe `metadata` pour avoir les infos nécessaires sur ces échantillons, qui sera plus simple à manipuler.

In [ ]:
## Code cell 13 ##

metadata <- data.frame(
  sample_id = metadataTai_wt$sample_name,
  study = "Tai-2022",
  rnaseq = "stranded_PE", 
  genotype = as.character(metadataTai_wt$genotype),
  tissue = metadataTai_wt$source_name,  
  sex = metadataTai_wt$gender,
  replicate = as.integer(substr(metadataTai_wt$sample_name,nchar(metadataTai_wt$sample_name),nchar(metadataTai_wt$sample_name))),
  stringsAsFactors = FALSE
)

metadata

A l'issue du chargement des données, on peut lister les éléments chargés dans notre session, et supprimer ceux qui ne sont plus utiles.

In [ ]:
## Code cell 14 ##

ls()

In [ ]:
## Code cell 15 ##

rm(countsTai, metadataTai, metadataTai_wt, names)

-------

## 2. Exploration initiale des données brutes

### 2.A - Distribution des reads counts

Avant toute normalisation, nous examinons la distribution des read counts bruts.

In [ ]:
## Code cell 16 ##

dim(countsTai_wt)
#summary(countsTot)

# Statistiques descriptives des read counts
summary(colSums(countsTai_wt[ ,3:18], na.rm=TRUE))
summary(rowSums(countsTai_wt[ ,3:18], na.rm=TRUE))


<div class="alert alert-block alert-info"> <b> Question 1 : </b><br>
<b> - Que pensez vous de cette distribution ? <br>
    - Les données vous semblent-elles être brutes ou normalisées ? </b> </div>
___ (ajoutez une cellule Markdown en dessous pour taper votre réponse)

In [ ]:
## Code cell 17 ##

options(repr.plot.width = 10, repr.plot.height = 6)

# create a colour vector to color samples by tissue
tissueColor <- match(metadata$tissue, c("Cortex", "Liver")) + 1
# '+1' to avoid color '1' i.e. black

# Visualisation de la distribution des read counts totaux par échantillon
barplot(colSums(countsTai_wt[ ,3:18], na.rm=TRUE)/1e6, 
        las = 2, 
        main = "Nombre total de reads par échantillon",
        ylab = "Millions de reads",
        col = tissueColor)
legend("topright", 
       legend = levels(factor(metadata$tissue)),
       fill = 2:3)


<div class="alert alert-block alert-info"> <b> Question 2 : </b><br>
<b> - Quelle est l'origine de la variation observée entre les échantillons ? <br>
    - Comment cette variabilité va t'elle être prise en compte dans la suite de l'analyse ? </b> </div>
___ 

### 2.B - Préfiltrage des gènes faiblement exprimés

Nous retirons les gènes avec très peu de reads pour améliorer la qualité de l'analyse et réduire le temps de calcul. 
Pour déterminer le seuil à utiliser, il est courant de prendre le nombre d'échantillons - 1. Enlever tous les gènes qui ont un read count total inférieur au nombre d'échantillon veut dire qu'aucun echantillon du dataset ne présente une expression au-delà de quelques reads.   

Ici, nous avons 16 échantillons, nous prenons donc un seuil de 15.

Un seuil minimal de 15 reads au total sur tous les échantillons est appliqué.

In [ ]:
## Code cell 18 ##

# Filtrer les gènes avec moins de 10 reads au total
keep <- rowSums(countsTai_wt[ ,3:18], na.rm=TRUE) >= 15
counts_filtered <- countsTai_wt[keep, ]

#summary(keep)

# Afficher le nombre de gènes conservés
cat("Gènes avant filtrage:", nrow(countsTai_wt), "\n")
cat("Gènes après filtrage:", nrow(counts_filtered), "\n")
cat("Gènes retirés:", sum(!keep), "\n")


<div class="alert alert-block alert-info"> <b> Question 3 : </b><br>
<b> - Est-ce que le nombre de gènes éliminés vous semble important ? Logique ? <br>
    - Pourquoi ? </b> </div>
___ 

In [ ]:
## Code cell 19 ##

dim(counts_filtered)
head(counts_filtered)

In [ ]:
## Code cell 20 ##

options(repr.plot.width = 10, repr.plot.height = 6)

# create a colour vector to color samples by tissue
tissueColor <- match(metadata$tissue, c("Cortex", "Liver")) + 1
# '+1' to avoid color '1' i.e. black

# Visualisation de la distribution des read counts filtrés par échantillon
barplot(colSums(counts_filtered[ ,3:18], na.rm=TRUE)/1e6, 
        las = 2, 
        main = "Nombre total de reads par échantillon",
        ylab = "Millions de reads",
        col = tissueColor)
legend("topright", 
       legend = levels(factor(metadata$tissue)),
       fill = 2:3)


<div class="alert alert-block alert-info"> <b> Question 4 : </b><br>
<b> - Ce graphe est-il différent de celui obtenu avec la cellule 17 ? <br>
    - Pourquoi ? </b> </div>
___ 

Il est très souvent utilie de visualiser la distribution des valeurs d'expression avec un boxplot : 

In [ ]:
## Code cell 21 ##

options(repr.plot.width = 10, repr.plot.height = 6)

# create a colour vector to color samples by tissue
tissueColor <- match(metadata$tissue, c("Cortex", "Liver")) + 1
# '+1' to avoid color '1' i.e. black

# Visualisation de la distribution des read counts filtrés par échantillon
boxplot(counts_filtered[ ,3:18]/1e6, na.rm=TRUE, 
        las = 2,    # allows to change the orientation of the axis labels: 2 is for always perpendicular
        main = "Expression des gènes avant normalisation",
        ylab = "Millions de reads",
        col = tissueColor)
legend("topright", 
       legend = levels(factor(metadata$tissue)),
       fill = 2:3)


<div class="alert alert-block alert-info"> <b> Question 5 : </b><br>
<b> - Pourquoi les distributions sont-elles tassées vers le bas ? <br>
    - A quoi correspondent les ronds isolés au-dessus ? </b> </div>
___ 

Comme on s'y attend, il y a beaucoup de très faibles valeurs, on choisit donc de faire ce boxplot avec les valeurs d'expression passées en log2 : 

In [ ]:
## Code cell 22 ##

options(repr.plot.width = 10, repr.plot.height = 6)

# create a colour vector to color samples by tissue
tissueColor <- match(metadata$tissue, c("Cortex", "Liver")) + 1
# '+1' to avoid color '1' i.e. black

# Visualisation de la distribution des read counts filtrés par échantillon
boxplot(log2(counts_filtered[ ,3:18]+1), 
        las = 2,    # allows to change the orientation of the axis labels: 2 is for always perpendicular
        main = "Expression des gènes avant normalisation",
        ylab = "log2(counts + 1)",
        col = tissueColor)
legend("topright", 
       legend = levels(factor(metadata$tissue)),
       fill = 2:3)


<div class="alert alert-block alert-info"> <b> Question 6 : </b><br>
<b> - A quoi correspondent les "boites" sur le graphes ? <br>
    - Les données vous semblent-elles être brutes ou normalisées ? </b> </div>
___ 

### 2.C - Analyse en Composantes Principales

#### a - Calcul de la PCA  

L'Analyse en Composantes Principales (ACP, ou PCA en anglais) permet de visualiser la structure globale des données et d'identifier les principales sources de variation entre les échantillons. Nous commençons par réaliser cette analyse sur toutes les données filtrées.

In [ ]:
## Code cell 23 ##

# run PCA
PCAdata <- prcomp(t(counts_filtered[, -c(1:2)])) # we get rid of the first two columns of counts_filtered that do not contain count data
summary(PCAdata)

#### b - Diagrammes des éboulis

Les diagrammes des éboulis permettent de visualiser la relation entre les composantes de la PCA (ou "axes") et la part de la variance totale qu'elles représentent : 

In [ ]:
## Code cell 24 ##

options(repr.plot.width = 12, repr.plot.height = 6)

# to display the two scree plots side by side
layout(matrix(1:2, ncol = 2))

screeplot(PCAdata) # barplot representation
screeplot(PCAdata, type = "lines", main = "Screeplot PCAdata - Eigenvalues") # same but with a line

#### c - Plots de la PCA 

La visualisation des axes de la PCA permet de voir comment se répartissent les échantillons, et si une explication biologique soutend la part de variance expliquée par chaque composante.

In [ ]:
## Code cell 25 ##

options(repr.plot.width = 12, repr.plot.height = 8)

autoplot(PCAdata,
         data = metadata, 
         colour = "sex", 
         shape = "tissue",
         size = 4) +
        geom_text_repel(aes(x = PC1, y = PC2, label = sample_id), box.padding = 0.3, max.overlaps = 15) +
        labs(title = "PCA 1 & 2 des échantillons non normalisés")


In [ ]:
## Code cell 26 ##

options(repr.plot.width = 12, repr.plot.height = 8)

autoplot(PCAdata,
         x = 2,    # PC2
         y = 3,    # PC3
         data = metadata, 
         colour = "sex", 
         shape = "tissue",
         size = 4) +
        geom_text_repel(aes(x = PC2, y = PC3, label = sample_id), box.padding = 0.3, max.overlaps = 15) +
        labs(title = "PCA 2 & 3 des échantillons non normalisés")



<div class="alert alert-block alert-info"> <b> Question 7 : </b><br>
<b> - Quelle part de la variance total est-elle expliquée par la PC1 ? par la PC2 ? Par la PC3 ? <br>
    - Vous semblent-elles correspondre à une variation biologique ? Technique ? </b> </div>
___ 

### 2.D - PCA sur les gènes les plus variables   

#### a - Tri des gènes

Par définition, seuls les gènes dont l'expression varie contribuent à la différence entre les échantillons (variabilité biologique au sein des conditions) et à la différence entre les conditions testées (la différence qui nous intéresse).    
Par conséquent, la bonne pratique est de faire une PCA uniquement sur les gènes les plus variables. Nous allons donc sélectionner les gènes sur leur variance, puis refaire l'analyse PCA sur ceux-là seulement.

In [ ]:
## Code cell 27

variances <- apply(counts_filtered[, -c(1:2)], 1, var)
sorted_var <- sort(variances, decreasing = TRUE)
top_300_genes <- names((sorted_var)[1:300])

On extrait les données des 300 gènes les plus variables :

In [ ]:
## Code cell 28

counts_filtered300 <- counts_filtered[top_300_genes, ]
dim(counts_filtered300)
head(counts_filtered300)

#### b - Calcul et plots de la PCA 

On refait l'analyse PCA sur les 300 gènes les plus variables : 

In [ ]:
## Code cell 29 ##

# run PCA
PCAdata <- prcomp(t(counts_filtered300[, -c(1:2)])) # we get rid of the first two columns of counts_filtered that do not contain count data
summary(PCAdata)

##### Diagrammes des éboulis

In [ ]:
## Code cell 30 ##

options(repr.plot.width = 10, repr.plot.height = 5)

# to display the two scree plots side by side
layout(matrix(1:2, ncol = 2))

screeplot(PCAdata) # barplot representation
screeplot(PCAdata, type = "lines", main = "Screeplot sur les 300 gènes les plus variables") # same but with a line

##### Plots de la PCA 

In [ ]:
## Code cell 31 ##

options(repr.plot.width = 12, repr.plot.height = 8)

autoplot(PCAdata,
         data = metadata, 
         colour = "sex", 
         shape = "tissue",
         size = 4) +
        geom_text_repel(aes(x = PC1, y = PC2, label = sample_id), box.padding = 0.3, max.overlaps = 15) +
        labs(title = "PCA 1 & 2 des counts non normalisés - 300 top genes")


In [ ]:
## Code cell 32 ##

options(repr.plot.width = 12, repr.plot.height = 8)

autoplot(PCAdata,
         x = 2,    # PC2
         y = 3,    # PC3
         data = metadata, 
         colour = "sex", 
         shape = "tissue",
         size = 4) +
        geom_text_repel(aes(x = PC2, y = PC3, label = sample_id), box.padding = 0.3, max.overlaps = 15) +
        labs(title = "PCA 2 & 3 des counts non normalisés - 300 top genes")



<div class="alert alert-block alert-info"> <b> Question 8 : </b><br>
<b> - La sélection des gènes les plus variables a t-elle changé vos conclusions à la suite de la question 7 ? <br>
    - D'après-vous pourquoi ? </b> </div>
___ 

### 2.E - Clustering hiérarchique

Il est également possible de visualiser la relation entre les échantillons à l'aide d'un clustering hiérachique.
On peut en tracer un avec les données filtrées, mais pas encore normalisées. 

In [ ]:
## Code cell 33 ##

options(repr.plot.width = 8, repr.plot.height = 5)


sampleDists <- dist(t(as.matrix(counts_filtered[, -c(1:2)])))

sampleDistMatrix <- as.matrix(sampleDists)
rownames(sampleDistMatrix) <- paste(metadata$sample_id)
colnames(sampleDistMatrix) <- NULL
colors <- colorRampPalette(rev(brewer.pal(7, "Blues")) )(255)
pheatmap(sampleDistMatrix,
         clustering_distance_rows=sampleDists,
         clustering_distance_cols=sampleDists,
         col=colors)


_____

## 3. Création de l'objet DESeq2

Pour poursuivre l'analyse de nos données, il faut maintenant les normaliser, afin de corriger les variations entre comptages de reads qui ne sont pas dûes à des différence biologiques, mais à des paramètres expérimentaux : profondeurs de séquençage variables, faible détection des gènes faiblement exprimés, etc...   

Un des outils les plus utilisés en analyse RNAseq bulk est **DESeq2**.  
Il travaille à partir d'un objet R qui lui est propre, *DESeqDataSet*, qui est en général nommé `dds`, et qui stocke l'ensemble des données et des informations nécessaires à l'analyse. 

Nous créons l'objet DESeqDataSet avec les comptages, les métadonnées et le modèle statistique. Le modèle statistique inclut la ou les sources attendues de variations entre les échantillons. Ici dans notre cas, on suppose des variations selon le type de tissu et selon le sexe des souris comme facteurs principaux.

In [ ]:
## Code cell 34 ##

# Préparer les informations des échantillons pour DESeq2
# et indiquer quels facteurs utiliser, avec leurs niveaux 
coldata <- data.frame(
    row.names = metadata$sample_name,
    tissue = factor(metadata$tissue, levels = c("Cortex", "Liver")),
    sex = factor(metadata$sex, levels = c("male", "female"))
)

# Afficher le résumé
summary(coldata)

In [ ]:
## Code cell 35 ##

# Créer l'objet DESeqDataSet
dds <- DESeqDataSetFromMatrix(countData = counts_filtered[ ,3:18],
                               colData = coldata,
                               design = ~ tissue + sex + tissue:sex)


cat("Données chargées et préparées\n")


Nous vérifions le nombre de genes dans le dataset

In [ ]:
## Code cell 36 ##

nrow(dds)

Et on affiche l'objet dds :

In [ ]:
## Code cell 37 ##

dds

Dans `dds`, les comptages sont rangés dans la portion "assays", qui est accessible avec la fonction `assay()` de DESeq2 : 

In [ ]:
## Code cell 38 ##

head(assay(dds))

## 4. Normalisation des read counts

Les données de comptage présentes dans `assay(dds)`sont des nombres entiers : ce sont des données brutes non normalisées. 

Nous appliquons une transformation pour stabiliser la variance des comptages :
**vst** (variance stabilizing transformation), rapid et adaptée aux grands jeux de données

Cette transformation est nécessaire pour les analyses exploratoires comme l'ACP, car elles rendent les données plus homoscédastiques (variance constante quel que soitke niveau d'expression).

In [ ]:
## Code cell 39 ##

# Transformation vst
vsd <- vst(dds, blind = TRUE)

cat("Transformation vst terminée\n")

On peut voir l'effet de la normalisation dans `assay(vsd)` :

In [ ]:
## Code cell 40 ##

head(assay(vsd))


<div class="alert alert-block alert-info"> <b> Question 9 : </b><br>
<b> - A quoi voyez-vous que les données ont été normalisées ? <br>
 </b> </div>
___ 

## 5. Visualisation des données normalisées 

### 5.A - Boxplot

Le même boxplot que précedemment sur les données normalisées : 

In [ ]:
## Code cell n°41 ##

options(repr.plot.width = 8, repr.plot.height = 6)

# create a colour vector to color samples by tissue
tissueColor <- match(metadata$tissue, c("Cortex", "Liver")) + 1
# '+1' to avoid color '1' i.e. black

# Visualisation de la distribution des read counts filtrés par échantillon
boxplot(assay(vsd), na.rm=TRUE, 
        las = 1,    # allows to change the orientation of the axis labels: 1 is for always horizontal
        main = "Expression des gènes après normalisation",
        ylab = "Niveau d'expression",
        col = tissueColor)
legend("topright", 
       legend = levels(factor(metadata$tissue)),
       fill = 2:3)


<div class="alert alert-block alert-info"> <b> Question 10 : </b><br>
<b> - A quoi voyez-vous que les données ont été normalisées ? <br>
 </b> </div>
___ 

### 5.B - Clustering hiérarchique

On refait un clustering hiérachique avec les données normalisées par la fonction `vst`. 

In [ ]:
## Code cell 42 ##

sampleDists <- dist(t(assay(vsd)))

sampleDistMatrix <- as.matrix(sampleDists)
rownames(sampleDistMatrix) <- paste(metadata$sample_id)
colnames(sampleDistMatrix) <- NULL
colors <- colorRampPalette( rev(brewer.pal(9, "Blues")) )(255)
pheatmap(sampleDistMatrix,
         clustering_distance_rows=sampleDists,
         clustering_distance_cols=sampleDists,
         col=colors)


### 5.C - PCA avec DESeq2

La librairie `DESeq2` permet de faire directement une PCA sur des données RNA-seq.
Attention, toutefois :
- par défaut, seuls les 2 premiers axes sont sauvegardés.
- seuls les gènes les plus variables sont utilisés (par défaut : 500)

Par conséquence, les résultats seront différents d'une PCA "classique" qui aurait pris un nombre différent de top genes.

In [ ]:
## Code cell 43 ##

pcaData <- DESeq2::plotPCA(vsd, 
                   intgroup=c("tissue"), 
                   returnData=TRUE)

head(pcaData)
attr(pcaData, "percentVar")

percentVar <- round(100 * attr(pcaData, "percentVar"))

options(repr.plot.width = 8, repr.plot.height = 6)

ggplot(pcaData, aes(PC1, PC2, color=tissue)) +
  geom_point(size=3) +
  xlab(paste0("PC1: ",percentVar[1],"% variance")) +
  ylab(paste0("PC2: ",percentVar[2],"% variance")) + 
  theme_bw() +
  theme(legend.position="bottom", legend.key.height=unit(1,"line"), legend.key.width=unit(1,"line"))



## 6. Modélisation de la distribution des read counts

DESeq2 utilise un modèle linéaire généralisé (GLM) basé sur la distribution binomiale négative pour modéliser les read counts. Cette distribution est adaptée aux données de comptage qui présentent une surdispersion (variance supérieure à la moyenne).

Les étapes principales sont :
1. **Estimation des facteurs de taille** : normalisation pour les différences de profondeur de séquençage entre échantillons
2. **Estimation de la dispersion** : mesure de la variabilité biologique entre réplicats
3. **Test statistique** : identification des gènes différentiellement exprimés  

La fonction `DESeq()` exécute l'analyse DESeq2 complète avec les 3 étapes.   
Patientez, elle peut prendre un peu de temps. 

In [ ]:
## Code cell 44 ##

# Cette fonction effectue les trois étapes : normalisation, estimation de la dispersion, et tests statistiques
dds2 <- DESeq2::DESeq(dds)

cat("Analyse DESeq2 terminée\n")

Let's inspect the newly created object dds2. *(Note that in most tutorials, they often replace dds here. In fact, the dds object is completed with the results of the 3 steps. For clarity, we called the object dds2).*

In [ ]:
## Code cell 45 ##

dds2

The portion `assays` within `dds2` contains the expression data, still not normalized:

In [ ]:
## Code cell 46 ##

head(assay(dds2))

### Plot de dispersion   

DESeq2 utilise le *shrinkage* pour corriger les estimations de log fold change (LogFC) des gènes peu exprimés, dont les valeurs pourraient sinon être artificiellement extrêmes à cause du faible nombre de lectures.   

*Par exemple, un gène qui aurait environ 5 reads dans une condition et 15 reads dans une autre pourrait apparaître comme surexprimé 3 fois dans la seconde condition. Cependant, il paraît évident que cette différence de read counts ne représente pas une vraie surexpression d'un point de vue biologique.*     

Cette méthode rapproche les LogFC des faibles read counts de zéro, évitant ainsi qu’ils dominent les résultats. La significativité d’un gène dépend non seulement de son LogFC, mais aussi de sa variabilité au sein des groupes, mesurée par la dispersion : pour les gènes fortement exprimés, une dispersion de 0,01 signifie par exemple une variabilité typique de 10 % entre répliquats (√0,01), tandis que pour les gènes faiblement exprimés, le bruit de Poisson augmente cette dispersion. La fonction `plotDispEsts` permet de visualiser ces estimations et l'effet de l'étape de *shrinkage*, aidant à évaluer la fiabilité des résultats.

In [ ]:
## Code cell 47 ##

# for figure display
options(repr.plot.width = 12, repr.plot.height = 8) 

# Visualiser la relation entre dispersion et moyenne d'expression
plotDispEsts(dds2, main = "Estimation de la dispersion" , ylim = c(1e-6, 1e2))

## 7. Analyse différentielle

### 7.A - Foie vs Cortex

Nous comparons les échantillons de Foie aux échantillons de Cortex pour identifier les gènes dont l'expression est modifiée par le type tissulaire.

In [ ]:
## Code cell 48 ##

# Extraire les résultats pour le contraste (comparaison) Foie vs Cortex
res <- results(dds2, contrast = c("tissue", "Liver", "Cortex"))

# Afficher un résumé
summary(res)

#### a - Utiliser un seuil différent : alpha 0.00001

Par défaut, la fonction `result` utilise un seuil de p-value ajustée (FDR) de 0.1. On peut utiliser un autre seuil de FDR, en définissant `alpha` de façon explicite.   
Le résultat précédent indique un nombre très important de gènes avec un LogFC non nul, on peut donc tester l'effet de diminuer le taux de faux positifs, avec `alpha=0,00001`.

In [ ]:
## Code cell 49 ##

res2 <- results(dds2, contrast = c("tissue", "Liver", "Cortex"),  alpha = 0.0001)
summary(res2)

#### b - Trier les gènes DE

In [ ]:
## Code cell 50 ##

# Ordonner les résultats par p-valeur ajustée
res2_ordered <- res2[order(res2$padj), ]

# Afficher les 10 gènes les plus significatifs
head(res2_ordered, 10)

#### c - Ajouter les noms de gènes   

On voit que les gènes sont identifiés par leur numéros, mais il est nécessaire d'avoir les noms de gènes pour la suite. On ajoute donc les noms des gènes dans le dataframe de résultats. 

In [ ]:
## Code cell 51 ##

# 1. Convertir DESeqResults en data.frame
res2_df <- as.data.frame(res2)

# 2. Extraire les IDs des gènes différentiellement exprimés
de_gene_ids <- rownames(res2_df)

# 3. Créer un vecteur de noms de gènes correspondant aux IDs de res2
#    (en utilisant match() pour associer chaque ID à son nom)
gene_names <- counts_filtered$gene_name[match(de_gene_ids, rownames(counts_filtered))]

# 4. Ajouter la colonne gene_name au dataframe
res2_names <- cbind(res2_df, gene_name = gene_names)

# 5. Réorganiser les colonnes pour mettre gene_name au début
res2_names <- res2_names %>%
  dplyr::select(gene_name,baseMean, log2FoldChange, lfcSE, stat, pvalue, padj)

# 6. Vérifier qu'il n'y a pas de NA dans la colonne gene_name
sum(is.na(res2_names$gene_name))  # Doit retourner 0

# 7. Afficher les 10 premières lignes
head(res2_names, 10)


In [ ]:
## Code cell 52 ##

# Ordonner les résultats par p-valeur ajustée
res2_ordered_names <- res2_names[order(res2_names$padj), ]

# Afficher les 10 gènes les plus significatifs
head(res2_ordered_names, 6)

#### d - Sélectionner et afficher les gènes DE significatifs : Volcano Plot 

On sélectionne les gènes DE à la fois sur la p-value (ajustée, correction des tests multiples par la méthode Benjamini-Hochberg), et par le FC minimal.   
Rappel : 1 point de log2 FC correspond à une expression doublée ou réduite de moitié.   

In [ ]:
## Code cell 53 ##

# Compter le nombre de gènes différentiellement exprimés (seuils : padj < 0.0001 et |log2FC| > 3)
sig_genes <- subset(res2_names, padj < 0.000001 & abs(log2FoldChange) > 4)

cat("Nombre de gènes différentiellement exprimés (Foie vs Cortex):", nrow(sig_genes), "\n")
cat("  - Sur-exprimés:", sum(sig_genes$log2FoldChange >= 4), "\n")
cat("  - Sous-exprimés:", sum(sig_genes$log2FoldChange <= -4), "\n")

In [ ]:
## Code cell 54 ##

# for figure display in the notebook
options(repr.plot.width = 15, repr.plot.height = 8) 

# Volcano plot pour visualiser les résultats
res2_names_df <- as.data.frame(res2_names)
res2_names_df$significant <- ifelse(res2_names_df$padj < 0.00001 & abs(res2_names_df$log2FoldChange) > 4, 
                                  "Significatif", "Non significatif")

ggplot(res2_names_df, aes(x = log2FoldChange, y = -log10(padj), color = significant)) +
    geom_point(alpha = 0.5, size = 2) +
    scale_color_manual(values = c("Non significatif" = "grey", "Significatif" = "red")) +
    geom_vline(xintercept = 4, linetype = "dashed", color = "red") +
    geom_vline(xintercept = -4, linetype = "dashed", color = "blue") +
    geom_hline(yintercept = -log10(0.00001), linetype = "dashed", color = "green") +
    labs(title = "Volcano plot - Foie vs Cortex",
         x = "log2 Fold Change",
         y = "-log10(p-valeur ajustée)") +
    theme_minimal()

Grâce au volcano plot, nous voyons que nous obtenons beaucoup de gènes DE car notre seuil de log2 FC est trop bas.   
On modifie donc notre seuil, et on refait le volcano plot. On en profite pour ajouter les noms des gènes gardés comme significativement DE. 

In [ ]:
## Code cell 55 ##

# Compter le nombre de gènes différentiellement exprimés (seuils : padj < 0.00001 et |log2FC| > 4)
sig <- subset(res2_names, padj < 0.00001 & abs(log2FoldChange) > 10)

cat("Nombre de gènes différentiellement exprimés (Foie vs Cortex):", nrow(sig), "\n")
cat("  - Sur-exprimés:", sum(sig$log2FoldChange >= 10), "\n")
cat("  - Sous-exprimés:", sum(sig$log2FoldChange <= -10), "\n")

In [ ]:
## Code cell 56 ##

# for figure display in the notebook
options(repr.plot.width = 15, repr.plot.height = 8) 

# Volcano plot pour visualiser les résultats
res2_names_df <- as.data.frame(res2_names)
res2_names_df$significant <- ifelse(res2_names_df$padj < 0.00001 & abs(res2_names_df$log2FoldChange) > 10, 
                                  "Significatif", "Non significatif")

ggplot(res2_names_df, aes(x = log2FoldChange, y = -log10(padj), color = significant)) +
    geom_point(alpha = 0.5, size = 2) +
    scale_color_manual(values = c("Non significatif" = "grey", "Significatif" = "red")) +
    geom_vline(xintercept = 10, linetype = "dashed", color = "red") +
    geom_vline(xintercept = -10, linetype = "dashed", color = "blue") +
    geom_hline(yintercept = -log10(0.00001), linetype = "dashed", color = "green") +
    # Ajouter les noms des gènes différentiellement exprimés (sig_dup)
    ggrepel::geom_text_repel(
        data = sig_genes,  # Utiliser directement le sous-ensemble sig_genes
        aes(label = gene_name),  # Étiqueter avec le nom du gène
        box.padding = 0.5,  # Éviter les chevauchements
        max.overlaps = 15,  # augmente le nombre de points étiquetés
        size = 4,  # Taille du texte
        color = "black"  # Couleur du texte
    ) +
    labs(title = "Volcano plot - Foie vs Cortex",
         x = "log2 Fold Change",
         y = "-log10(p-valeur ajustée)") +
    theme_minimal()

### 7.B - Mâles vs Femelles   

Nous avons vu grâce à la PCA qu'une petite part de la variance est expliquée par le sexe des souris.   
Nous allons donc refaire l'analyse DE sur ce facteur, identifier les gènes DE correspondant et refaire le volcano plot pertinent.   


<div class="alert alert-block alert-info"> <b> Question 11 : </b><br>
<b> - Au vu de la PCA, attendez-vous beaucoup ou peu de gènes DE selon ce facteur ? <br>
 </b> </div>
___ 

In [ ]:
## Code cell 57 ##

# Extraire les résultats pour le contraste (comparaison) Female vs Male
res_sex <- results(dds2, contrast = c("sex", "female", "male"))

# Afficher un résumé
summary(res_sex)

#### a - Utiliser un seuil différent : alpha 0.2
 
Le résultat précédent indique un nombre très faible de gènes avec un LogFC non nul, on peut donc tester l'effet de modifier le FDR, avec `alpha=0,2`.

In [ ]:
## Code cell 58 ##

res2_sex <- results(dds2, contrast = c("sex", "female", "male"),  alpha = 0.2)
summary(res2_sex)

#### b - Trier les gènes DE

In [ ]:
## Code cell 59 ##

# Ordonner les résultats par p-valeur ajustée
res2_sex_ordered <- res2_sex[order(res2_sex$padj), ]

# Afficher les 4 gènes les plus significatifs
head(res2_sex_ordered, 4)

#### c - Ajouter les noms de gènes   

In [ ]:
## Code cell 60 ##

# 1. Convertir DESeqResults en data.frame
res2_sex_df <- as.data.frame(res2_sex)

# 2. Extraire les IDs des gènes différentiellement exprimés
de_gene_ids <- rownames(res2_sex_df)

# 3. Créer un vecteur de noms de gènes correspondant aux IDs de res2_sex
#    (en utilisant match() pour associer chaque ID à son nom)
gene_names <- counts_filtered$gene_name[match(de_gene_ids, rownames(counts_filtered))]

# 4. Ajouter la colonne gene_name au dataframe
res2_sex_names <- cbind(res2_sex_df, gene_name = gene_names)

# 5. Réorganiser les colonnes pour mettre gene_name au début
res2_sex_names <- res2_sex_names %>%
  dplyr::select(gene_name,baseMean, log2FoldChange, lfcSE, stat, pvalue, padj)

# 6. Vérifier qu'il n'y a pas de NA dans la colonne gene_name
sum(is.na(res2_sex_names$gene_name))  # Doit retourner 0

# 7. Afficher les 4 premières lignes
head(res2_sex_names, 4)

In [ ]:
## Code cell 61 ##

# Ordonner les résultats par p-valeur ajustée
res2_sex_ordered_names <- res2_sex_names[order(res2_sex_names$padj), ]

# Afficher les 10 gènes les plus significatifs
head(res2_sex_ordered_names, 10)


<div class="alert alert-block alert-info"> <b> Question 12 : </b><br>
<b> - Quel gène vous semble logiquement DE entre les femelles et les mâles ? <br>
    - Le sens du Fold Change vous semble t'il logique ?
 </b> </div>
___ 

#### d - Sélectionner et afficher les gènes DE significatifs : Volcano Plot 

Nous utilisons des seuils beaucoup moins stringents que pour la première analyse DE. 

In [ ]:
## Code cell 62 ##

# Compter le nombre de gènes différentiellement exprimés (seuils : padj < 0.05 et |log2FC| > 1)
sig_genes_sex <- subset(res2_sex_names, padj < 0.05 & abs(log2FoldChange) > 1)

cat("Nombre de gènes différentiellement exprimés (Female vs Male):", nrow(sig_genes), "\n")
cat("  - Sur-exprimés:", sum(sig_genes$log2FoldChange >= 1), "\n")
cat("  - Sous-exprimés:", sum(sig_genes$log2FoldChange <= -1), "\n")

In [ ]:
## Code cell 63 ##

# for figure display in the notebook
options(repr.plot.width = 15, repr.plot.height = 8) 

# Volcano plot pour visualiser les résultats
res2_sex_names_df <- as.data.frame(res2_sex_names)
res2_sex_names_df$significant <- ifelse(res2_sex_names_df$padj < 0.05 & abs(res2_sex_names_df$log2FoldChange) > 1, 
                                  "Significatif", "Non significatif")

ggplot(res2_sex_names_df, aes(x = log2FoldChange, y = -log10(padj), color = significant)) +
    geom_point(alpha = 0.5, size = 2) +
    scale_color_manual(values = c("Non significatif" = "grey", "Significatif" = "red")) +
    geom_vline(xintercept = 1, linetype = "dashed", color = "red") +
    geom_vline(xintercept = -1, linetype = "dashed", color = "blue") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "green") +
    # Ajouter les noms des gènes différentiellement exprimés (sig_dup)
    ggrepel::geom_text_repel(
        data = sig_genes_sex,  # Utiliser directement le sous-ensemble sig_dup
        aes(label = gene_name),  # Étiqueter avec le nom du gène
        box.padding = 0.5,  # Éviter les chevauchements
        max.overlaps = 20,  # augmente le nombre de points étiqutés
        size = 6,  # Taille du texte
        color = "black"  # Couleur du texte
    ) +
    labs(title = "Volcano plot - Female vs Male",
         x = "log2 Fold Change",
         y = "-log10(p-valeur ajustée)") +
    theme_minimal()

## 8. Sauvegarde des résultats

Nous sauvegardons les tableaux de résultats pour une analyse ultérieure.

In [ ]:
## Code cell 64 ##
dim(res2_ordered_names)
head(res2_ordered_names) 
dim(sig_genes)
head(sig_genes)

dim(res2_sex_ordered_names)
head(res2_sex_ordered_names) 
dim(sig_genes_sex)
head(sig_genes_sex)

In [ ]:
## Code cell 65 ##

# Sauvegarder les résultats complets
write.table(as.data.frame(res2_ordered_names), 
            file = "DESeq2_Liver_vs_Cortex_allgenes.txt",
            sep = "\t", quote = FALSE, row.names = TRUE)

write.table(as.data.frame(res2_sex_ordered_names), 
            file = "DESeq2_Female_vs_Male_allgenes.txt",
            sep = "\t", quote = FALSE, row.names = TRUE)

# Sauvegarder uniquement les gènes significatifs
write.table(as.data.frame(sig_genes), 
            file = "DESeq2_Liver_vs_Cortex_significant.txt",
            sep = "\t", quote = FALSE, row.names = TRUE)

write.table(as.data.frame(sig_genes_sex), 
            file = "DESeq2_Female_vs_Male_significant.txt",
            sep = "\t", quote = FALSE, row.names = TRUE)

cat("Résultats sauvegardés\n")

Maj 17/02/2026 - SCaburet